<a href="https://colab.research.google.com/github/prabdeepkaur/NEURAL-NETWORK/blob/main/assign2_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **ANN**
Take a dataset from Kaggle and perform preprocessing using ANN and CNN concept?


In [ ]:
import pandas as pd
import numpy as np
import os

# Load dataset
print(os.listdir(path))   # check file name

file_path = os.path.join(path, "titanic.csv")
df = pd.read_csv(file_path)

print(df.head())

['titanic.csv']
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1


In [ ]:
# Drop unwanted columns
df = df.drop(columns=['name', 'ticket', 'cabin'], errors='ignore')

# Fill missing values safely
if 'age' in df.columns:
    df['age'].fillna(df['age'].mean(), inplace=True)

if 'embarked' in df.columns:
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

# Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

if 'sex' in df.columns:
    df['sex'] = le.fit_transform(df['sex'])

if 'embarked' in df.columns:
    df['embarked'] = le.fit_transform(df['embarked'])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Target column (important)
y = df['Survived']
X = df.drop('Survived', axis=1)

# Scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

model.add(Dense(16, input_dim=X.shape[1], activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.4232 - loss: 0.7035 - val_accuracy: 0.4627 - val_loss: 0.6987
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5880 - loss: 0.6867 - val_accuracy: 0.7015 - val_loss: 0.6783
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6479 - loss: 0.6746 - val_accuracy: 0.7015 - val_loss: 0.6648
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6479 - loss: 0.6671 - val_accuracy: 0.7015 - val_loss: 0.6554
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6479 - loss: 0.6611 - val_accuracy: 0.7015 - val_loss: 0.6494
Epoch 6/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6479 - loss: 0.6577 - val_accuracy: 0.7015 - val_loss: 0.6435
Epoch 7/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6479 - loss: 0.6543 - val_accuracy: 0.7015 - val_loss: 0.6390
Epoch 8/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6479 - loss: 0.6518 - val_accuracy: 0.7015 - val_loss

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("ANN Accuracy:", acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5357 - loss: 0.7328
ANN Accuracy: 0.5357142686843872


# **CNN**

In [ ]:
import os
import pandas as pd

print(os.listdir(path))

['titanic.csv']


In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("francescomonny/plant-pathology")

print("PATH:", path)
print(os.listdir(path))

100%|██████████| 779M/779M [00:09<00:00, 84.1MB/s]

Extracting files...


PATH: /root/.cache/kagglehub/datasets/francescomonny/plant-pathology/versions/1
['train.csv', 'sample_submission.csv', 'images', 'test.csv']


In [ ]:
import pandas as pd
import os

csv_path = os.path.join(path, "train.csv")
df = pd.read_csv(csv_path)

print(df.head())

  image_id  healthy  multiple_diseases  rust  scab
0  Train_0        0                  0     0     1
1  Train_1        0                  1     0     0
2  Train_2        1                  0     0     0
3  Train_3        0                  0     1     0
4  Train_4        1                  0     0     0


In [ ]:
df['label'] = df[['healthy','multiple_diseases','rust','scab']].idxmax(axis=1)

print(df[['image_id','label']].head())

  image_id              label
0  Train_0               scab
1  Train_1  multiple_diseases
2  Train_2            healthy
3  Train_3               rust
4  Train_4            healthy


In [ ]:
image_dir = os.path.join(path, "images")

df['image_path'] = df['image_id'].apply(lambda x: os.path.join(image_dir, x + ".jpg"))

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

IMG_SIZE = 128

X = []
y = []

for i in range(len(df)):
    img = load_img(df['image_path'][i], target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img) / 255.0

    X.append(img)
    y.append(df['label'][i])

X = np.array(X)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y = le.fit_transform(y)
y = to_categorical(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 30s 778ms/step - accuracy: 0.3308 - loss: 1.5611 - val_accuracy: 0.4829 - val_loss: 1.1749
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 27s 734ms/step - accuracy: 0.4304 - loss: 1.1941 - val_accuracy: 0.4897 - val_loss: 1.1435
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 27s 737ms/step - accuracy: 0.5163 - loss: 1.1118 - val_accuracy: 0.5034 - val_loss: 1.1189
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 28s 772ms/step - accuracy: 0.5696 - loss: 1.0218 - val_accuracy: 0.4212 - val_loss: 1.2394
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 41s 755ms/step - accuracy: 0.6718 - loss: 0.8341 - val_accuracy: 0.4212 - val_loss: 1.2519
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 40s 754ms/step - accuracy: 0.7809 - loss: 0.6274 - val_accuracy: 0.4384 - val_loss: 1.3684
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 28s 753ms/step - accuracy: 0.8660 - loss: 0.3864 - val_accuracy: 0.4692 - val_loss: 1.6577
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 41s 746ms/step - accuracy: 0.9416 - loss: 0.2052 - val_accu

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("CNN Accuracy:", acc)

12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 184ms/step - accuracy: 0.4822 - loss: 2.3510
CNN Accuracy: 0.4821917712688446
